[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/47_lora_data.ipynb)

#  Medium: LoRA Data Preparation

Prepare data specifically formatted for **LoRA (Low-Rank Adaptation)** fine-tuning with proper prompt/completion splitting and label masking.

### Signature
```python
def prepare_lora_data(data, tokenizer, max_length=512, prompt_template=None):
    """Prepare data for LoRA fine-tuning.
    Args:
        data: List[{"instruction": str, "completion": str}]
        tokenizer: HuggingFace-style tokenizer
        max_length: max sequence length
        prompt_template: str with {instruction} and {completion} placeholders
    Returns:
        dict with keys "input_ids" (B, L) and "labels" (B, L)
            labels: prompt tokens = -100, completion tokens = original IDs
    """
    ...
```

### Rules
- Apply `prompt_template` to format each sample (default: `"{instruction}\n\n{completion}"`)
- Tokenize the full prompt+completion text
- Compute prompt length by tokenizing prompt-only, mask those positions as -100
- Truncate or pad to `max_length`
- Return batched tensors with proper padding

### Example
```python
>>> data = [{"instruction": "Say hello", "completion": "Hello!"}]
>>> result = prepare_lora_data(data, tokenizer, max_length=64)
>>> result["labels"][0, :5]  # prompt region
tensor([-100, -100, -100, -100, -100])
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  YOUR IMPLEMENTATION HERE

def prepare_lora_data(data, tokenizer, max_length=512, prompt_template=None):
    """Prepare data for LoRA fine-tuning."""
    pass  # Replace this


In [ ]:
#  Test your implementation

class SimpleTokenizer:
    def __init__(self):
        self.pad_token_id = 0
        self.eos_token_id = 2
    def __call__(self, text, return_tensors=None, padding=False, truncation=False, max_length=None):
        ids = [ord(c) for c in text] + [self.eos_token_id]
        if max_length and len(ids) > max_length:
            ids = ids[:max_length-1] + [self.eos_token_id]
        return {"input_ids": torch.tensor(ids), "attention_mask": [1]*len(ids)}

tok = SimpleTokenizer()
data = [{"instruction": "Hi", "completion": "Hello"}]
result = prepare_lora_data(data, tok, max_length=64)

print("input_ids shape:", result["input_ids"].shape)
print("labels:", result["labels"])

# First part of labels (prompt) should be -100
labels = result["labels"][0]
prompt_token_count = len("Hi\n\n") + sum(1 for _ in "Hi\n\n") + 1  # chars + eos
print(f"Prompt ends at position ~{prompt_token_count}")
assert "input_ids" in result and "labels" in result
print(" Passed!")

In [ ]:
#  SUBMIT  Run this cell to check your solution
from torch_judge import check
check("lora_data")